In [1]:
import sys
sys.path.append(r'd:\VSCode\neylon-ai\primary-server')

In [2]:
import os
print(os.getcwd())

d:\VSCode\neylon-ai\primary-server\resume_assistant


In [ ]:
pip install PyMuPDF

In [7]:
import fitz

path = 'lib/data/Hruthik_m.pdf'
print(os.path.exists(path))

True


In [10]:
pip install --upgrade PyMuPDF


Note: you may need to restart the kernel to use updated packages.


In [20]:
doc = fitz.open(path)
text = ""
for page in doc:
    text += page.get_text("text")
    # for b in blocks:
    #     text += b[4].strip() + "\n\n"

with open("lib/data/output.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Text with layout preserved saved to output.txt")

✅ Text with layout preserved saved to output.txt


In [34]:
doc = fitz.open(path)
text = ""
for page in doc:
    text += page.get_text("text")
    # for b in blocks:
    #     text += b[4].strip() + "\n\n"

print(text)
with open("lib/data/output.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Text with layout preserved saved to output.txt")

Hruthik M   
+91 7483229386 |  mhrithik450@gmail.com | LinkedIn | GitHub    
   
Experience    
  
 
Software Development Engineer 
   
 April 2025 – Present    
Next.js Developer  | Codedale    
    
   
 India   
- 
Built and maintained 100+ robust backend endpoints using TypeScript and Next.js, reducing API 
response time by 90% and improving system reliability for 100k users   
- 
Designed and implemented PostgreSQL schemas and Drizzle ORM models handling 20K+ 
records, optimizing queries and reducing average database load by 60%.  
- 
Contributed to 50+ production features end-to-end, including API design, authentication, and 
data validation, supporting 10k+ active users.  
 
Founder & Lead Engineer 
   
 Aug 2025 – Present    
Next.js And Django Developer  | Neylon AI  
    
   
 India   
- 
Founded Neylon AI, an AI agency delivering scalable AI assistants and agent-based solutions, 
serving diverse clients with intelligent automation.   
- 
Developed AI agents using LangGraph, 

In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY=os.getenv("GOOGLE_API_KEY")

base_model = ChatGoogleGenerativeAI(
    model='gemini-2.5-pro',
    temperature=0.4,
    max_retries=2,
    google_api_key=GEMINI_API_KEY
)

In [39]:
SYSTEM_PROMPT="""
You are an expert Resume Role Adaptation Assistant. Your goal is to intelligently tailor an existing resume to match a given job role or job description without losing structure, section order, or factual accuracy.

Rules:
0. Preserve Original Structure:
   - Keep the resume’s sections in the same order as the original.
   - Keep all section titles exactly as they appear
1. Selective Modification Based on Role:
   - Adjust the section lines under each experience/project to align with the provided job role.
   - Only make modifications necessary to improve relevance (e.g., emphasizing specific technologies, soft skills, or responsibilities that match the job role).
   - Do not invent or remove any real experience, project, or education unless explicitly instructed by the user.
2. Preserve Authenticity:
   - Keep all factual details (names, dates, titles, metrics, and projects) intact unless the user provides new information.
   - Do not alter quantitative achievements or company names.
3. Integrate Additional Content if Provided:
   - If the user provides extra material, seamlessly insert it into the most relevant section — without breaking the structure.
4. Tone and Style:
   - Maintain a professional, concise, and impact-driven tone.
   - Use action verbs and role-aligned keywords relevant to the target job description.
   - Focus on clarity and alignment with the target role’s requirements.
5. Output Format:
   - Return the entire modified resume text.
   - Ensure section headings, bullet structure, and layout are clearly preserved.
   - No explanations, comments, or notes — only the final formatted resume.

Input Format Expected:
- Resume Extracted Text: (full existing resume in plain text)
- Target Role or Job Description: (details of the new role)
- (Optional) Additional Info / Projects: any new content to integrate.

Generate a fully rewritten resume optimized for the target job role while preserving all original sections, order, and authenticity.
"""

In [40]:
import tiktoken 

encoding_model = "cl100k_base"
def get_encoding(text: str)->int:
    encoding = tiktoken.get_encoding(encoding_model)
    return len(encoding.encode(text))

In [41]:
print(get_encoding(SYSTEM_PROMPT))

390


In [46]:
user_prompt="""
Job Title: Full Stack Developer
Company: Cloudify Tech
Location: Remote

Description:
We are seeking a Full Stack Developer proficient in JavaScript and modern web frameworks. The ideal candidate will have hands-on experience developing scalable backend APIs, secure authentication systems, and interactive UIs.

Additional project:

CloudMetrics Dashboard
- Designed and developed a cloud monitoring dashboard using Next.js and Express.
- Integrated AWS CloudWatch APIs for real-time data visualization.
- Deployed application using Docker on AWS EC2.
"""

In [52]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"old_resume: {text}\nuser: {user_prompt}"}
]

In [53]:
response = base_model.invoke(messages)

In [56]:
with open("lib/data/response.txt", "w", encoding="utf-8") as f:
    f.write(response.content)

In [ ]:
pip install reportlab

In [58]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
import textwrap

output_filename = "lib/data/final_resume.pdf"
resume_text_path = "lib/data/response.txt"  # The text response file from your AI
font_name = "Helvetica"

with open(resume_text_path, "r", encoding="utf-8") as f:
    resume_text = f.read()

pdf = SimpleDocTemplate(
    output_filename,
    pagesize=A4,
    rightMargin=50,
    leftMargin=50,
    topMargin=50,
    bottomMargin=50
)

styles = getSampleStyleSheet()
normal = styles["Normal"]
normal.fontName = font_name
normal.fontSize = 11
normal.leading = 15

heading_style = ParagraphStyle(
    name="Heading",
    parent=styles["Heading2"],
    fontSize=14,
    textColor=colors.HexColor("#2E86C1"),
    spaceAfter=10,
    spaceBefore=10
)

content = []

lines = resume_text.split("\n")
for line in lines:
    line = line.strip()
    if not line:
        content.append(Spacer(1, 10))
        continue

    # Detect section headers (simple heuristic)
    if line.isupper() and len(line.split()) <= 5:
        content.append(Paragraph(f"<b>{line}</b>", heading_style))
    else:
        # Wrap long text for clean formatting
        wrapped = "<br/>".join(textwrap.wrap(line, width=90))
        content.append(Paragraph(wrapped, normal))

pdf.build(content)

print(f"✅ PDF created successfully: {output_filename}")

✅ PDF created successfully: lib/data/final_resume.pdf


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer
from reportlab.lib import colors

resume_data = {
    "name": "Hruthik M",
    "contact": {
        "phone": "+91 7483229386",
        "email": "mhrithik450@gmail.com",
        "links": {
            "LinkedIn": "linkedin.com/in/hruthik-m-3595a0329",
            "GitHub": "github.com/Hrithik450"
        }
    },
    "sections": [
        {
            "title": "Experience",
            "entries": [
                {
                    "role": "Software Development Engineer",
                    "position": "Next.js Developer",
                    "company": "Codedale",
                    "duration": "April 2025 – Present",
                    "location": "India",
                    "highlights": [
                        "Built and maintained 100+ robust backend endpoints using TypeScript and Next.js, reducing API response time by 90% and improving system reliability for 100k users.",
                        "Designed and implemented PostgreSQL schemas and Drizzle ORM models handling 20K+ records, optimizing queries and reducing average database load by 60%.",
                        "Contributed to 50+ production features end-to-end, including API design, authentication, and data validation, supporting 10k+ active users."
                    ]
                },
                {
                    "role": "Founder & Lead Engineer",
                    "position": "Next.js And Django Developer",
                    "company": "Neylon AI",
                    "duration": "Aug 2025 – Present",
                    "location": "India",
                    "highlights": [
                        "Founded Neylon AI, an AI agency delivering scalable AI assistants and agent-based solutions, serving diverse clients with intelligent automation.",
                        "Developed AI agents using LangGraph, LangChain, and vector databases, implementing a RAG pipeline capable of handling 20K+ data records efficiently.",
                        "Engineered the platform to process and query 10GB+ datasets seamlessly using Django backend and Next.js frontend, ensuring high-performance AI workflows."
                    ]
                }
            ]
        }
    ]
}

# PDF setup
pdf_file = "lib/data/Hruthik_M_Resume.pdf"
doc = SimpleDocTemplate(pdf_file, pagesize=A4, rightMargin=10, leftMargin=10, topMargin=10, bottomMargin=10)

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="Name", fontSize=32, alignment=1, spaceAfter=8, leading=36, fontName="Times-Roman"))
styles.add(ParagraphStyle(name="Contact", fontSize=12, alignment=1, spaceAfter=20, leading=14, fontName="Times-Roman"))
styles.add(ParagraphStyle(name="SectionTitle", fontSize=14, leading=16, spaceBefore=10, spaceAfter=6, underlineWidth=1))
styles.add(ParagraphStyle(name="JobTitle", fontSize=11, leading=14, spaceBefore=6, spaceAfter=2, leftIndent=10))
styles.add(ParagraphStyle(name="CustomBullet", fontSize=10, leftIndent=20, spaceBefore=2, spaceAfter=2, bulletIndent=10))

story = []

# Name
story.append(Paragraph(resume_data["name"], styles["Name"]))

# Contact
contact_info = (
    f"{resume_data['contact']['phone']} |"
    f"<link href='mailto:{resume_data['contact']['email']}' color='blue'>{resume_data['contact']['email']}</link> |"
    f"<link href='{resume_data['contact']['links']['LinkedIn']}' color='blue'>linkedIn</link> |"
    f"<link href='{resume_data['contact']['links']['GitHub']}' color='blue'>gitHub</link>"
)
story.append(Paragraph(contact_info, styles["Contact"]))

# Experience Section
for section in resume_data["sections"]:
    story.append(Paragraph(section["title"], styles["SectionTitle"]))
    story.append(Spacer(1, 6))

    for entry in section["entries"]:
        job_header = f'<b>{entry["role"]}</b><br/>{entry["position"]} | {entry["company"]}'
        duration_location = f'<para alignment="right"><b>{entry["duration"]}</b><br/>{entry["location"]}</para>'
        story.append(Paragraph(job_header, styles["JobTitle"]))
        story.append(Paragraph(duration_location, styles["Normal"]))

        for point in entry["highlights"]:
            story.append(Paragraph(f'• {point}', styles["Bullet"]))
        story.append(Spacer(1, 10))

doc.build(story)
print(f"✅ Resume saved as {pdf_file}")

✅ Resume saved as lib/data/Hruthik_M_Resume.pdf
